# 01B — Fresh Re-audit of the Clean Split

**NO TRAINING.**

This notebook performs a fresh image-similarity scan after rebuilding the dataset.

Checks:
- SHA-256 exact duplicates across Train / Validation / Test
- pHash nearest-neighbour search across partitions
- SIFT + RANSAC verification for suspicious pairs

**Training is allowed only if the final confirmed cross-partition near-duplicate count is zero.**

In [1]:
import sys, subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'opencv-python-headless'], check=True)

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, hashlib, multiprocessing as mp
import cv2, numpy as np, pandas as pd

PROJECT = Path('/content/drive/MyDrive/Cataract')
ROOT = PROJECT / 'Data_Clean_LeakageControlled'
OUT = PROJECT / 'FINAL_REVISION_2026_08' / 'clean_split_audit' / 'fresh_reaudit'
OUT.mkdir(parents=True, exist_ok=True)

CLASS_ORDER = ['Cataract','Normal','Not Eye']
SPLITS = ['Train','Validation','Test']
assert ROOT.exists(), f'Run Notebook 01 first: {ROOT}'

Mounted at /content/drive


In [2]:
def pack64(bits):
    return np.packbits(bits.astype(np.uint8).reshape(-1)).tobytes().hex()

def phash(gray):
    small=cv2.resize(gray,(32,32),interpolation=cv2.INTER_AREA).astype(np.float32)
    D=cv2.dct(small)[:8,:8]
    med=np.median(D.flatten()[1:])
    return pack64(D.flatten()>med)

def fingerprint(p):
    data=p.read_bytes()
    sha=hashlib.sha256(data).hexdigest()
    arr=np.frombuffer(data,np.uint8)
    img=cv2.imdecode(arr,cv2.IMREAD_COLOR)
    if img is None:
        return None
    gray=cv2.cvtColor(img,cv2.COLOR_BGR2GRAY)
    rel=p.relative_to(ROOT).as_posix()
    split, cls, fn = rel.split('/',2)
    return {
        'path':rel,'split':split,'class':cls,'filename':fn,
        'sha256':sha,'phash64':phash(gray)
    }

paths=[p for s in SPLITS for c in CLASS_ORDER for p in (ROOT/s/c).glob('*') if p.is_file()]
print('Images:',len(paths))
rows=[]
for i,p in enumerate(paths,1):
    r=fingerprint(p)
    if r: rows.append(r)
    if i%1000==0: print(i,'/',len(paths))
df=pd.DataFrame(rows)
df.to_csv(OUT/'fingerprints_clean.csv',index=False)
print(df.groupby(['split','class']).size())

Images: 13639
1000 / 13639
2000 / 13639
3000 / 13639
4000 / 13639
5000 / 13639
6000 / 13639
7000 / 13639
8000 / 13639
9000 / 13639
10000 / 13639
11000 / 13639
12000 / 13639
13000 / 13639
split       class   
Test        Cataract     857
            Normal       979
            Not Eye      756
Train       Cataract    2932
            Normal      3348
            Not Eye     2584
Validation  Cataract     722
            Normal       825
            Not Eye      636
dtype: int64


In [3]:
# Exact duplicates across any two partitions
bad=[]
for sha,g in df.groupby('sha256'):
    if g['split'].nunique()>1:
        bad.append(g)
cross_exact=pd.concat(bad,ignore_index=True) if bad else pd.DataFrame(columns=df.columns)
cross_exact.to_csv(OUT/'cross_partition_exact_duplicates.csv',index=False)
print('Exact cross-partition duplicate rows:',len(cross_exact))

Exact cross-partition duplicate rows: 0


In [4]:
# pHash nearest neighbour comparisons:
# compare Validation -> Train,
# Test -> Train,
# Test -> Validation.

LUT=np.array([bin(i).count('1') for i in range(256)],dtype=np.uint8)
def to_u64(x): return np.uint64(int(x,16))
def hdist(arr,x):
    v=np.bitwise_xor(arr,np.uint64(x))
    return LUT[v.view(np.uint8).reshape(-1,8)].sum(axis=1)

df['ph_u']=df.phash64.map(to_u64)

comparisons=[('Validation','Train'),('Test','Train'),('Test','Validation')]
cand_rows=[]

for qsplit, rsplit in comparisons:
    q=df[df.split==qsplit].reset_index(drop=True)
    r=df[df.split==rsplit].reset_index(drop=True)
    rph=r.ph_u.to_numpy(np.uint64)
    for _,x in q.iterrows():
        d=hdist(rph,x.ph_u)
        # keep best same-class and best any-class candidates
        same=(r['class'].to_numpy()==x['class'])
        if same.any():
            ids=np.where(same)[0]
            j=ids[np.argmin(d[same])]
            cand_rows.append({
                'query_path':x.path,'query_split':qsplit,'query_class':x['class'],
                'ref_path':r.iloc[j].path,'ref_split':rsplit,'ref_class':r.iloc[j]['class'],
                'phash_distance':int(d[j]),'candidate_type':'same_class'
            })
        j=int(np.argmin(d))
        cand_rows.append({
            'query_path':x.path,'query_split':qsplit,'query_class':x['class'],
            'ref_path':r.iloc[j].path,'ref_split':rsplit,'ref_class':r.iloc[j]['class'],
            'phash_distance':int(d[j]),'candidate_type':'nearest_any'
        })

cands=pd.DataFrame(cand_rows)
cands.to_csv(OUT/'phash_candidates_clean.csv',index=False)
print('Candidates with pHash <=12:',int((cands.phash_distance<=12).sum()))

Candidates with pHash <=12: 2028


In [5]:
# Verify suspicious candidates with SIFT + RANSAC.
susp=cands[cands.phash_distance<=12].drop_duplicates(['query_path','ref_path']).copy()

def prep(rel):
    im=cv2.imread(str(ROOT/rel),cv2.IMREAD_GRAYSCALE)
    h,w=im.shape
    scale=min(1.0,512/max(h,w))
    if scale<1:
        im=cv2.resize(im,(int(w*scale),int(h*scale)),interpolation=cv2.INTER_AREA)
    return im

def sift_one(rec):
    a=prep(rec['query_path']); b=prep(rec['ref_path'])
    sift=cv2.SIFT_create(nfeatures=1200,contrastThreshold=0.02)
    k1,x1=sift.detectAndCompute(a,None); k2,x2=sift.detectAndCompute(b,None)
    good=[]; inl=0; ratio=0.0
    if x1 is not None and x2 is not None and len(x1)>=2 and len(x2)>=2:
        bf=cv2.BFMatcher(cv2.NORM_L2)
        matches=bf.knnMatch(x1,x2,k=2)
        good=[p for p,q in matches if p.distance < 0.75*q.distance]
        if len(good)>=4:
            src=np.float32([k1[m.queryIdx].pt for m in good]).reshape(-1,1,2)
            dst=np.float32([k2[m.trainIdx].pt for m in good]).reshape(-1,1,2)
            H,mask=cv2.findHomography(src,dst,cv2.RANSAC,5.0)
            if mask is not None:
                inl=int(mask.sum()); ratio=inl/len(good)
    out=dict(rec)
    out.update({
        'sift_good':len(good),
        'sift_inliers':inl,
        'sift_inlier_ratio':ratio,
        'confirmed_strict': bool(len(good)>=20 and inl>=15 and ratio>=0.5)
    })
    return out

verified=[]
for i,rec in enumerate(susp.to_dict('records'),1):
    verified.append(sift_one(rec))
    if i%100==0: print(i,'/',len(susp))

ver=pd.DataFrame(verified)
ver.to_csv(OUT/'sift_verified_clean.csv',index=False)

confirmed=ver[ver.confirmed_strict] if len(ver) else ver
confirmed.to_csv(OUT/'CONFIRMED_CROSS_PARTITION_NEAR_DUPLICATES.csv',index=False)

print('\nConfirmed strict cross-partition near-duplicate pairs:',len(confirmed))
if len(cross_exact)==0 and len(confirmed)==0:
    print('\nPASS ✅ CLEAN SPLIT VERIFIED. You may proceed to Notebook 02.')
else:
    print('\nSTOP ❌ Do not train yet. Review the confirmed pair CSV and rebuild once more.')

100 / 1120
200 / 1120
300 / 1120
400 / 1120
500 / 1120
600 / 1120
700 / 1120
800 / 1120
900 / 1120
1000 / 1120
1100 / 1120

Confirmed strict cross-partition near-duplicate pairs: 514

STOP ❌ Do not train yet. Review the confirmed pair CSV and rebuild once more.
